# Multi-Flow Mobile Social-Media Traffic Classification Benchmark

This version uses a strict temporal holdout: **Day 5 is completely held out for testing**. Days 1–4 are used for development, with **90% of Days 1–4 for training and 10% for validation**. Multi-flow windows remain **[1, 10, 20, 30, 40, 50, 60] flows**.


Models: XGBoost, LightGBM, HistGradientBoosting, CatBoost, and Random Forest.



In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
!pip install -q lightgbm xgboost catboost shap lime

In [ ]:
import random, warnings
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.preprocessing import LabelEncoder, OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestClassifier, HistGradientBoostingClassifier
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    classification_report, confusion_matrix, log_loss
)
import lightgbm as lgb
import xgboost as xgb
from catboost import CatBoostClassifier

warnings.filterwarnings('ignore')

DATA_DIR = Path('/content/drive/MyDrive/mobile_merged_output')
RESULTS_DIR = Path('/content/drive/MyDrive/mobile_day5_holdout_results_5models_main')
for name in ['metrics','per_class','confusion_matrices','plots','predictions']:
    (RESULTS_DIR/name).mkdir(parents=True, exist_ok=True)

SEEDS = [42,123,456,789,2026]
WINDOW_SIZES = [1, 10, 20, 30, 40, 50, 60]
RARE_MIN_COUNT = 10

## Load data

In [ ]:
files = sorted([p for p in DATA_DIR.glob('*.csv') if 'summary' not in p.name.lower()])
if not files:
    raise FileNotFoundError(f'No CSV files found in {DATA_DIR}')

frames = []
for p in files:
    d = pd.read_csv(p)
    if 'Label' not in d.columns:
        d['Label'] = p.stem.lower()
    if 'capture_file' not in d.columns:
        raise ValueError(f'{p.name} has no capture_file column')
    d['_original_row_order'] = np.arange(len(d))
    frames.append(d)
    print(f'{p.name}: {len(d):,} rows')

df = pd.concat(frames, ignore_index=True)
print('Combined:', df.shape)
display(df['Label'].value_counts().sort_index())

## Day 5 held-out evaluation

Day 5 is kept completely unseen for final testing.

Days 1–4 are the development data:
- 90% of Days 1–4 → training
- 10% of Days 1–4 → validation
- 100% of Day 5 → test

The 10% validation sample is stratified by application label. Day 5 is never used for
training, preprocessing fitting, rare-category fitting, imputation fitting, or validation.
Multi-flow windows are formed only during testing and never cross capture-file boundaries.


In [ ]:
from sklearn.model_selection import train_test_split

# Normalize capture_day so values such as 5, '5', 'day5', or 'Day_5' can be recognized.
if 'capture_day' not in df.columns:
    raise ValueError(
        "The dataset has no 'capture_day' column. "
        "Day 5 cannot be held out without an explicit day identifier."
    )

def normalize_day(value):
    s = str(value).strip().lower()
    digits = ''.join(ch for ch in s if ch.isdigit())
    return int(digits) if digits else np.nan

df['_day_number'] = df['capture_day'].map(normalize_day)

print('Detected capture days:')
display(df['_day_number'].value_counts(dropna=False).sort_index())

if 5 not in set(df['_day_number'].dropna().astype(int).unique()):
    raise ValueError(
        "Day 5 was not detected in capture_day. "
        "Inspect the printed capture_day values before continuing."
    )

# Day 5 is the completely unseen test set.
test_mask = df['_day_number'].eq(5)

# Only Days 1–4 are allowed into model development.
development_mask = df['_day_number'].isin([1, 2, 3, 4])

if (~(test_mask | development_mask)).any():
    unexpected = sorted(df.loc[~(test_mask | development_mask), 'capture_day'].astype(str).unique())
    raise ValueError(
        f'Unexpected capture_day values found: {unexpected}. '
        'This notebook expects Days 1–5 only.'
    )

development_idx = np.where(development_mask)[0]
test_idx = np.where(test_mask)[0]

# Use 10% of Days 1–4 for validation, stratified by application.
dev_labels = df.iloc[development_idx]['Label'].to_numpy()

train_idx, val_idx = train_test_split(
    development_idx,
    test_size=0.10,
    random_state=42,
    stratify=dev_labels
)

df['_split'] = 'unused'
df.loc[train_idx, '_split'] = 'train'
df.loc[val_idx, '_split'] = 'validation'
df.loc[test_idx, '_split'] = 'test'

# Preserve chronological ordering inside every capture file for later multi-flow windows.
order_candidates = [
    'bidirectional_first_seen_ms',
    'src2dst_first_seen_ms',
    'dst2src_first_seen_ms',
    'id',
    '_original_row_order'
]
ORDER_COL = next((c for c in order_candidates if c in df.columns), None)
if ORDER_COL is None:
    raise ValueError('No usable flow-order column was found.')

df['_flow_order_in_capture'] = -1
for capture, idx in df.groupby('capture_file', sort=False).groups.items():
    ordered_idx = (
        df.loc[idx]
          .sort_values([ORDER_COL, '_original_row_order'])
          .index
    )
    df.loc[ordered_idx, '_flow_order_in_capture'] = np.arange(len(ordered_idx))

print('\nSplit counts:')
display(df['_split'].value_counts())

print('\nSplit percentages of the complete dataset:')
display((df['_split'].value_counts(normalize=True) * 100).round(2))

print('\nClass counts by split:')
display(df.groupby(['Label', '_split']).size().unstack(fill_value=0))

print('\nDays present in each split:')
for split_name in ['train', 'validation', 'test']:
    days = sorted(df.loc[df['_split'].eq(split_name), '_day_number'].dropna().unique().tolist())
    print(f'{split_name}: {days}')

assert set(df.loc[df['_split'].eq('test'), '_day_number'].unique()) == {5}
assert 5 not in set(df.loc[df['_split'].isin(['train','validation']), '_day_number'].unique())

print('\nDay 5 holdout verified successfully.')


In [ ]:
# Count actual complete non-overlapping Day-5 windows without crossing capture files.
day5_meta = df.loc[df['_split'].eq('test'), ['capture_file', '_flow_order_in_capture']].copy()

print('Actual number of complete Day-5 windows available by size:')
for w in WINDOW_SIZES:
    total = 0
    for _, g in day5_meta.groupby('capture_file', sort=False):
        total += len(g) // w
    print(f'{w:>2} flow(s): {total:,} windows')


## Leakage-resistant feature preparation

In [ ]:
DROP_COLUMNS = {
    'Label','capture_day','capture_file','_day_number','_original_row_order','_split',
    '_flow_order_in_capture','id','expiration_id','application_name',
    'application_category_name','application_is_guessed','application_confidence',
    'requested_server_name','src_ip','dst_ip','src_mac','dst_mac','src_oui','dst_oui',
    'src_port','dst_port','splt_direction','splt_ps','splt_piat_ms'
}
ABSOLUTE_TIME_COLUMNS = {
    c for c in df.columns
    if c.endswith('_first_seen_ms') or c.endswith('_last_seen_ms')
}
DROP_COLUMNS |= ABSOLUTE_TIME_COLUMNS

X = df[[c for c in df.columns if c not in DROP_COLUMNS]].copy()
X = X.replace([np.inf,-np.inf], np.nan)

train_mask = df['_split'].eq('train')
train_view = X.loc[train_mask]
bad = list(train_view.columns[train_view.isna().all()])
bad += [c for c in train_view.columns if train_view[c].nunique(dropna=False) <= 1]
X = X.drop(columns=sorted(set(bad)))

le = LabelEncoder()
y = le.fit_transform(df['Label'])
class_names = le.classes_.tolist()
num_classes = len(class_names)

numeric_cols = X.select_dtypes(exclude=['object','string','category']).columns.tolist()
categorical_cols = [c for c in X.columns if c not in numeric_cols]

print('Feature matrix:', X.shape)
print('Numeric:', len(numeric_cols))
print('Categorical:', len(categorical_cols))
print('Classes:', class_names)

## Preprocessing and models

In [ ]:
class RareCategoryGrouper(BaseEstimator, TransformerMixin):
    def __init__(self, min_count=10, rare_token='__RARE__'):
        self.min_count = min_count
        self.rare_token = rare_token
    def fit(self, X, y=None):
        a = np.asarray(X, dtype=object)
        self.keep_ = []
        for j in range(a.shape[1]):
            s = pd.Series(a[:,j]).astype(str)
            vc = s.value_counts(dropna=False)
            self.keep_.append(set(vc[vc >= self.min_count].index))
        return self
    def transform(self, X):
        a = np.asarray(X, dtype=object).copy()
        for j, keep in enumerate(self.keep_):
            s = pd.Series(a[:,j]).astype(str)
            a[:,j] = np.where(s.isin(keep), s, self.rare_token)
        return a
    def get_feature_names_out(self, input_features=None):
        return np.asarray(input_features, dtype=object)

def make_preprocessor():
    return ColumnTransformer([
        ('num', Pipeline([
            ('imputer', SimpleImputer(strategy='median'))
        ]), numeric_cols),
        ('cat', Pipeline([
            ('imputer', SimpleImputer(strategy='most_frequent')),
            ('rare', RareCategoryGrouper(RARE_MIN_COUNT)),
            ('onehot', OneHotEncoder(handle_unknown='ignore', sparse_output=False))
        ]), categorical_cols)
    ], sparse_threshold=0)

def build_models(seed):
    return {
        'XGBoost': xgb.XGBClassifier(
            n_estimators=800, learning_rate=0.04, max_depth=6,
            min_child_weight=2, gamma=0.02, subsample=0.85,
            colsample_bytree=0.85, reg_alpha=0.05, reg_lambda=1.5,
            random_state=seed, n_jobs=-1, eval_metric='mlogloss',
            tree_method='hist', verbosity=0
        ),
        'LightGBM': lgb.LGBMClassifier(
            n_estimators=800, learning_rate=0.04, num_leaves=31,
            max_depth=8, min_child_samples=20, subsample=0.85,
            subsample_freq=1, colsample_bytree=0.85, reg_alpha=0.05,
            reg_lambda=1.5, random_state=seed, n_jobs=-1,
            class_weight='balanced', verbose=-1
        ),
        'HistGradientBoosting': HistGradientBoostingClassifier(
            max_iter=700, learning_rate=0.05, max_leaf_nodes=31,
            max_depth=8, min_samples_leaf=20, l2_regularization=1.0,
            early_stopping=True, validation_fraction=0.15,
            n_iter_no_change=30, random_state=seed
        ),
        'CatBoost': CatBoostClassifier(
            iterations=800, learning_rate=0.05, depth=7,
            l2_leaf_reg=3.0, random_strength=0.5,
            random_seed=seed, verbose=0, allow_writing_files=False
        ),
        'RandomForest': RandomForestClassifier(
            n_estimators=600, max_depth=15, min_samples_split=8,
            min_samples_leaf=3, max_features='sqrt',
            random_state=seed, n_jobs=-1,
            class_weight='balanced_subsample'
        )
    }

## Multi-flow probability averaging

In [ ]:
def aggregate_windows(probabilities, true_labels, metadata, window_size):
    probs_out, y_out, rows = [], [], []
    m = metadata.reset_index(drop=True).copy()
    m['_p'] = np.arange(len(m))

    for capture, g in m.groupby('capture_file', sort=False):
        g = g.sort_values('_flow_order_in_capture')
        pos = g['_p'].to_numpy()

        for start in range(0, len(pos), window_size):
            idx = pos[start:start+window_size]
            if len(idx) < window_size:
                continue
            labels = np.asarray(true_labels)[idx]
            if len(np.unique(labels)) != 1:
                continue
            probs_out.append(np.asarray(probabilities)[idx].mean(axis=0))
            y_out.append(int(labels[0]))
            rows.append({
                'capture_file': capture,
                'window_size': window_size,
                'window_start_order': int(g.iloc[start]['_flow_order_in_capture']),
                'window_end_order': int(g.iloc[start+window_size-1]['_flow_order_in_capture'])
            })

    return np.asarray(probs_out), np.asarray(y_out, dtype=int), pd.DataFrame(rows)

def metric_row(y_true, probs):
    pred = probs.argmax(axis=1)
    return {
        'Accuracy': accuracy_score(y_true,pred),
        'Precision_macro': precision_score(y_true,pred,average='macro',zero_division=0),
        'Recall_macro': recall_score(y_true,pred,average='macro',zero_division=0),
        'F1_macro': f1_score(y_true,pred,average='macro',zero_division=0),
        'F1_weighted': f1_score(y_true,pred,average='weighted',zero_division=0),
        'LogLoss': log_loss(y_true,probs,labels=np.arange(num_classes)),
        'Windows': len(y_true)
    }

## Train all five models across five seeds

In [ ]:
train_idx = np.where(df['_split'].eq('train'))[0]
val_idx = np.where(df['_split'].eq('validation'))[0]
test_idx = np.where(df['_split'].eq('test'))[0]

Xtr_raw, Xv_raw, Xte_raw = X.iloc[train_idx], X.iloc[val_idx], X.iloc[test_idx]
ytr, yv, yte = y[train_idx], y[val_idx], y[test_idx]
meta_test = df.iloc[test_idx][['capture_file','_flow_order_in_capture']].reset_index(drop=True)

metric_rows, class_rows, cms = [], [], {}

for seed in SEEDS:
    print('\nSEED', seed)
    np.random.seed(seed)
    random.seed(seed)

    prep = make_preprocessor()
    Xtr = prep.fit_transform(Xtr_raw)
    Xv = prep.transform(Xv_raw)
    Xte = prep.transform(Xte_raw)

    for model_name, model in build_models(seed).items():
        print('Training', model_name)

        if model_name == 'XGBoost':
            model.fit(Xtr,ytr,eval_set=[(Xv,yv)],verbose=False)
        elif model_name == 'LightGBM':
            model.fit(
                Xtr,ytr,eval_set=[(Xv,yv)],
                callbacks=[lgb.early_stopping(40, verbose=False)]
            )
        elif model_name == 'CatBoost':
            model.fit(Xtr,ytr,eval_set=(Xv,yv),early_stopping_rounds=40,verbose=False)
        else:
            model.fit(Xtr,ytr)

        flow_probs = model.predict_proba(Xte)

        for w in WINDOW_SIZES:
            wp, wy, wm = aggregate_windows(flow_probs,yte,meta_test,w)
            if len(wy) == 0:
                continue

            metric_rows.append({'Seed':seed,'Model':model_name,'Window_size':w,**metric_row(wy,wp)})
            pred = wp.argmax(axis=1)

            rep = classification_report(
                wy,pred,labels=np.arange(num_classes),
                target_names=class_names,output_dict=True,zero_division=0
            )
            for cls in class_names:
                class_rows.append({
                    'Seed':seed,'Model':model_name,'Window_size':w,'Class':cls,
                    'Precision':rep[cls]['precision'],
                    'Recall':rep[cls]['recall'],
                    'F1':rep[cls]['f1-score'],
                    'Support':rep[cls]['support']
                })

            cms.setdefault((model_name,w),[]).append(
                confusion_matrix(wy,pred,labels=np.arange(num_classes))
            )

            wm['true_class'] = [class_names[i] for i in wy]
            wm['predicted_class'] = [class_names[i] for i in pred]
            wm['seed'] = seed
            wm['model'] = model_name
            wm.to_csv(
                RESULTS_DIR/'predictions'/f'{model_name}_window{w}_seed{seed}.csv',
                index=False
            )

            print(f'  window={w}: acc={accuracy_score(wy,pred):.4f}, macroF1={f1_score(wy,pred,average="macro"):.4f}')

metrics_df = pd.DataFrame(metric_rows)
per_class_df = pd.DataFrame(class_rows)

metrics_df.to_csv(RESULTS_DIR/'metrics'/'all_seed_window_metrics.csv',index=False)
per_class_df.to_csv(RESULTS_DIR/'per_class'/'all_seed_window_per_class_metrics.csv',index=False)

## Mean ± standard deviation

In [ ]:
agg = metrics_df.groupby(['Model','Window_size']).agg(
    Accuracy_mean=('Accuracy','mean'), Accuracy_std=('Accuracy','std'),
    Precision_macro_mean=('Precision_macro','mean'), Precision_macro_std=('Precision_macro','std'),
    Recall_macro_mean=('Recall_macro','mean'), Recall_macro_std=('Recall_macro','std'),
    F1_macro_mean=('F1_macro','mean'), F1_macro_std=('F1_macro','std'),
    F1_weighted_mean=('F1_weighted','mean'), F1_weighted_std=('F1_weighted','std'),
    LogLoss_mean=('LogLoss','mean'), LogLoss_std=('LogLoss','std'),
    Windows_mean=('Windows','mean')
).reset_index()

for metric in ['Accuracy','Precision_macro','Recall_macro','F1_macro','F1_weighted','LogLoss']:
    agg[f'{metric}_mean_std'] = (
        agg[f'{metric}_mean'].map(lambda x:f'{x:.4f}') + ' ± ' +
        agg[f'{metric}_std'].fillna(0).map(lambda x:f'{x:.4f}')
    )

display_cols = [
    'Model','Window_size','Accuracy_mean_std','Precision_macro_mean_std',
    'Recall_macro_mean_std','F1_macro_mean_std','F1_weighted_mean_std',
    'LogLoss_mean_std','Windows_mean'
]
summary = agg[display_cols].sort_values(['Window_size','Model'])
summary.to_csv(RESULTS_DIR/'metrics'/'multiflow_summary_mean_std.csv',index=False)
display(summary)

## Performance curves

In [ ]:
for metric, ylabel, filename in [
    ('Accuracy','Mean accuracy','accuracy_by_window_size.png'),
    ('F1_macro','Mean macro F1','macro_f1_by_window_size.png')
]:
    pivot = metrics_df.groupby(['Window_size','Model'])[metric].mean().unstack()
    plt.figure(figsize=(10,6))
    for model_name in pivot.columns:
        plt.plot(pivot.index,pivot[model_name],marker='o',label=model_name)
    plt.xlabel('Number of consecutive flows')
    plt.ylabel(ylabel)
    plt.title(f'{ylabel} by Multi-Flow Window Size')
    plt.xticks(WINDOW_SIZES)
    plt.ylim(0,1)
    plt.legend()
    plt.tight_layout()
    plt.savefig(RESULTS_DIR/'plots'/filename,dpi=200)
    plt.show()

## Normalized mean confusion matrices

In [ ]:
for (model_name,w), matrices in cms.items():
    mean_cm = np.mean(np.stack(matrices),axis=0)
    row_sums = mean_cm.sum(axis=1,keepdims=True)
    pct = np.divide(mean_cm,row_sums,out=np.zeros_like(mean_cm,dtype=float),where=row_sums!=0)*100

    plt.figure(figsize=(11,9))
    im = plt.imshow(pct,cmap='viridis',vmin=0,vmax=100)
    plt.colorbar(im,label='Percentage (%)')
    plt.xticks(np.arange(num_classes),class_names,rotation=45,ha='right')
    plt.yticks(np.arange(num_classes),class_names)
    plt.xlabel('Predicted class')
    plt.ylabel('True class')
    plt.title(f'{model_name} - Mean Confusion Matrix (%) - {w} Flow(s)')
    for i in range(num_classes):
        for j in range(num_classes):
            plt.text(j,i,f'{pct[i,j]:.1f}%',ha='center',va='center',fontsize=7,
                     color='white' if pct[i,j]>=50 else 'black')
    plt.tight_layout()
    safe = model_name.replace(' ','_')
    plt.savefig(RESULTS_DIR/'confusion_matrices'/f'{safe}_window{w}_percent.png',dpi=200)
    plt.show()

    pd.DataFrame(pct,index=class_names,columns=class_names).to_csv(
        RESULTS_DIR/'confusion_matrices'/f'{safe}_window{w}_percent.csv'
    )

## Per-class mean and standard deviation

In [ ]:
per_class_summary = per_class_df.groupby(
    ['Model','Window_size','Class']
).agg(
    Precision_mean=('Precision','mean'), Precision_std=('Precision','std'),
    Recall_mean=('Recall','mean'), Recall_std=('Recall','std'),
    F1_mean=('F1','mean'), F1_std=('F1','std'),
    Support_mean=('Support','mean')
).reset_index()

per_class_summary.to_csv(
    RESULTS_DIR/'per_class'/'multiflow_per_class_mean_std.csv',
    index=False
)
display(per_class_summary)

## Improvement over the single-flow baseline

In [ ]:
mean_results = metrics_df.groupby(['Model','Window_size'])[['Accuracy','F1_macro']].mean().reset_index()
baseline = mean_results[mean_results.Window_size.eq(1)][['Model','Accuracy','F1_macro']].rename(
    columns={'Accuracy':'Baseline_accuracy','F1_macro':'Baseline_macro_F1'}
)
improvement = mean_results.merge(baseline,on='Model',how='left')
improvement['Accuracy_gain'] = improvement['Accuracy'] - improvement['Baseline_accuracy']
improvement['Macro_F1_gain'] = improvement['F1_macro'] - improvement['Baseline_macro_F1']
improvement.to_csv(RESULTS_DIR/'metrics'/'improvement_over_single_flow.csv',index=False)
display(improvement.sort_values(['Model','Window_size']))

## 13. Post-hoc explainability of the selected LightGBM model

The benchmark remains fair: all five models use identical preprocessing, Day-5 holdout,
and multi-flow evaluation. No L2 normalization or model-specific calibration is applied.

LightGBM is selected **after** the benchmark because it gives the strongest Day-5
performance at the 40-flow operating point. SHAP and LIME are then used only for
post-hoc explanation and do not alter predictions.

For each application class, this section reports the top 15 contributing features from:
- SHAP;
- LIME.

SHAP and LIME are computed on correctly classified Day-5 **individual flows** for the
selected LightGBM model. This keeps the explanations tied to the actual base model that
produces the probabilities later averaged in the multi-flow windows.


In [ ]:
import shap
from lime.lime_tabular import LimeTabularExplainer

XAI_TOP_K = 15
SHAP_BACKGROUND = 200
SHAP_SAMPLES_PER_CLASS = 100
LIME_SAMPLES_PER_CLASS = 25
LIME_NUM_SAMPLES = 3000
XAI_SEED = 42

XAI_DIR = RESULTS_DIR / 'xai'
XAI_SHAP_DIR = XAI_DIR / 'shap'
XAI_LIME_DIR = XAI_DIR / 'lime'
XAI_AGREE_DIR = XAI_DIR / 'agreement'

for p in [XAI_DIR, XAI_SHAP_DIR, XAI_LIME_DIR, XAI_AGREE_DIR]:
    p.mkdir(parents=True, exist_ok=True)

print('XAI output folder:', XAI_DIR)


In [ ]:
# Refit the selected LightGBM model deterministically for explanation.
# This uses the exact same train/validation/test protocol and preprocessing as the benchmark.
selected_seed = XAI_SEED

prep_xai = make_preprocessor()
Xtr_xai = prep_xai.fit_transform(Xtr_raw)
Xv_xai = prep_xai.transform(Xv_raw)
Xte_xai = prep_xai.transform(Xte_raw)

selected_lgbm = build_models(selected_seed)['LightGBM']
selected_lgbm.fit(
    Xtr_xai,
    ytr,
    eval_set=[(Xv_xai, yv)],
    callbacks=[lgb.early_stopping(40, verbose=False)]
)

feature_names = prep_xai.get_feature_names_out()
feature_names = np.asarray([str(x) for x in feature_names])

test_probs_xai = selected_lgbm.predict_proba(Xte_xai)
test_pred_xai = np.argmax(test_probs_xai, axis=1)

print('Selected LightGBM fitted.')
print('Transformed feature count:', len(feature_names))
print('Day-5 single-flow accuracy:',
      accuracy_score(yte, test_pred_xai))


In [ ]:
# Semantic feature mapping for fair SHAP-LIME comparison.
def semantic_feature_name(encoded_feature):
    name = str(encoded_feature)

    if name.startswith('num__'):
        return name[len('num__'):]

    if name.startswith('cat__'):
        remainder = name[len('cat__'):]
        for col in sorted(categorical_cols, key=len, reverse=True):
            if remainder == col or remainder.startswith(col + '_'):
                return col
        return remainder

    return name


def clean_feature_name(encoded_feature):
    name = str(encoded_feature)
    if name.startswith('num__'):
        return name[len('num__'):]
    if name.startswith('cat__'):
        return name[len('cat__'):]
    return name


In [ ]:
# SHAP: top 15 features for each class on correctly classified Day-5 flows.

rng = np.random.default_rng(XAI_SEED)

# Background drawn from training data only.
bg_n = min(SHAP_BACKGROUND, len(Xtr_xai))
bg_idx = rng.choice(len(Xtr_xai), size=bg_n, replace=False)
X_background = Xtr_xai[bg_idx]

explainer = shap.TreeExplainer(selected_lgbm)

shap_rows = []
shap_top_sets = {}
shap_semantic_sets = {}

for class_id, class_name in enumerate(class_names):
    candidates = np.where(
        (yte == class_id) &
        (test_pred_xai == class_id)
    )[0]

    if len(candidates) == 0:
        print(f'No correctly classified Day-5 flows for {class_name}; skipping SHAP.')
        shap_top_sets[class_name] = set()
        shap_semantic_sets[class_name] = set()
        continue

    take = min(SHAP_SAMPLES_PER_CLASS, len(candidates))
    chosen = rng.choice(candidates, size=take, replace=False)
    X_class = Xte_xai[chosen]

    shap_values = explainer.shap_values(X_class)

    # Handle LightGBM multiclass SHAP formats across shap versions.
    if isinstance(shap_values, list):
        class_shap = np.asarray(shap_values[class_id])
    else:
        arr = np.asarray(shap_values)
        if arr.ndim == 3:
            if arr.shape[2] == num_classes:
                class_shap = arr[:, :, class_id]
            elif arr.shape[0] == num_classes:
                class_shap = arr[class_id]
            else:
                raise ValueError(f'Unexpected SHAP array shape: {arr.shape}')
        elif arr.ndim == 2:
            class_shap = arr
        else:
            raise ValueError(f'Unexpected SHAP array shape: {arr.shape}')

    mean_abs = np.mean(np.abs(class_shap), axis=0)
    order = np.argsort(mean_abs)[::-1][:XAI_TOP_K]

    exact_set = set()
    semantic_set = set()

    for rank, feat_idx in enumerate(order, start=1):
        feat = feature_names[int(feat_idx)]
        semantic = semantic_feature_name(feat)

        exact_set.add(feat)
        semantic_set.add(semantic)

        shap_rows.append({
            'Class': class_name,
            'Rank': rank,
            'Feature': clean_feature_name(feat),
            'Encoded_feature': feat,
            'Semantic_feature': semantic,
            'Mean_abs_SHAP': float(mean_abs[int(feat_idx)]),
            'Explained_correct_flows': int(take),
        })

    shap_top_sets[class_name] = exact_set
    shap_semantic_sets[class_name] = semantic_set

shap_table = pd.DataFrame(shap_rows)
shap_table.to_csv(
    XAI_SHAP_DIR / 'lightgbm_shap_top15_per_class.csv',
    index=False
)

for cls in class_names:
    print('\n' + '=' * 100)
    print('SHAP -', cls)
    display(shap_table[shap_table['Class'] == cls])


In [ ]:
# LIME: top 15 features for each class on correctly classified Day-5 flows.
#
# Zero-inclusive aggregation:
# sum(abs(local weight)) / number of explained class instances.
# A feature absent from a local explanation contributes zero.

lime_explainer = LimeTabularExplainer(
    training_data=np.asarray(Xtr_xai),
    feature_names=feature_names.tolist(),
    class_names=class_names,
    mode='classification',
    discretize_continuous=True,
    random_state=XAI_SEED
)

lime_rows = []
lime_top_sets = {}
lime_semantic_sets = {}

for class_id, class_name in enumerate(class_names):
    candidates = np.where(
        (yte == class_id) &
        (test_pred_xai == class_id)
    )[0]

    if len(candidates) == 0:
        print(f'No correctly classified Day-5 flows for {class_name}; skipping LIME.')
        lime_top_sets[class_name] = set()
        lime_semantic_sets[class_name] = set()
        continue

    take = min(LIME_SAMPLES_PER_CLASS, len(candidates))
    chosen = rng.choice(candidates, size=take, replace=False)

    sums = {}
    occurrences = {}

    for idx in chosen:
        exp = lime_explainer.explain_instance(
            np.asarray(Xte_xai[idx]),
            selected_lgbm.predict_proba,
            labels=[class_id],
            num_features=XAI_TOP_K,
            num_samples=LIME_NUM_SAMPLES
        )

        for feat_idx, weight in exp.local_exp[class_id]:
            feat = feature_names[int(feat_idx)]
            sums[feat] = sums.get(feat, 0.0) + abs(float(weight))
            occurrences[feat] = occurrences.get(feat, 0) + 1

    averaged = {
        feat: total / take
        for feat, total in sums.items()
    }

    ranked = sorted(
        averaged.items(),
        key=lambda x: x[1],
        reverse=True
    )[:XAI_TOP_K]

    exact_set = set()
    semantic_set = set()

    for rank, (feat, importance) in enumerate(ranked, start=1):
        semantic = semantic_feature_name(feat)
        exact_set.add(feat)
        semantic_set.add(semantic)

        lime_rows.append({
            'Class': class_name,
            'Rank': rank,
            'Feature': clean_feature_name(feat),
            'Encoded_feature': feat,
            'Semantic_feature': semantic,
            'Mean_abs_LIME_zero_inclusive': float(importance),
            'Occurrence_count': int(occurrences.get(feat, 0)),
            'Explained_correct_flows': int(take),
            'Occurrence_rate': float(occurrences.get(feat, 0) / take),
        })

    lime_top_sets[class_name] = exact_set
    lime_semantic_sets[class_name] = semantic_set

lime_table = pd.DataFrame(lime_rows)
lime_table.to_csv(
    XAI_LIME_DIR / 'lightgbm_lime_top15_per_class.csv',
    index=False
)

for cls in class_names:
    print('\n' + '=' * 100)
    print('LIME -', cls)
    display(lime_table[lime_table['Class'] == cls])


In [ ]:
# SHAP-LIME agreement: exact encoded features and semantic/original feature families.

agreement_rows = []

for cls in class_names:
    shap_exact = shap_top_sets.get(cls, set())
    lime_exact = lime_top_sets.get(cls, set())
    shap_sem = shap_semantic_sets.get(cls, set())
    lime_sem = lime_semantic_sets.get(cls, set())

    exact_union = shap_exact | lime_exact
    semantic_union = shap_sem | lime_sem

    exact_jaccard = (
        len(shap_exact & lime_exact) / len(exact_union)
        if exact_union else np.nan
    )
    semantic_jaccard = (
        len(shap_sem & lime_sem) / len(semantic_union)
        if semantic_union else np.nan
    )

    agreement_rows.append({
        'Class': cls,
        'Exact_Jaccard': exact_jaccard,
        'Exact_common_count': len(shap_exact & lime_exact),
        'Semantic_Jaccard': semantic_jaccard,
        'Semantic_common_count': len(shap_sem & lime_sem),
        'Exact_common_features': '; '.join(sorted(shap_exact & lime_exact)),
        'Semantic_common_features': '; '.join(sorted(shap_sem & lime_sem)),
    })

agreement_df = pd.DataFrame(agreement_rows)
agreement_df.to_csv(
    XAI_AGREE_DIR / 'lightgbm_shap_lime_agreement_by_class.csv',
    index=False
)

display(agreement_df)

print('\nMean exact Jaccard:',
      agreement_df['Exact_Jaccard'].mean())
print('Mean semantic Jaccard:',
      agreement_df['Semantic_Jaccard'].mean())


In [ ]:
# Agreement graph.
x = np.arange(len(class_names))
width = 0.36

plt.figure(figsize=(12, 6))
plt.bar(
    x - width / 2,
    agreement_df['Exact_Jaccard'],
    width,
    label='Exact encoded features'
)
plt.bar(
    x + width / 2,
    agreement_df['Semantic_Jaccard'],
    width,
    label='Semantic feature families'
)

plt.xlabel('Application class')
plt.ylabel('Jaccard agreement')
plt.title('LightGBM SHAP-LIME Top-15 Agreement on Held-Out Day 5')
plt.xticks(x, class_names, rotation=45, ha='right')
plt.ylim(0, 1)
plt.legend()
plt.tight_layout()

plot_path = XAI_AGREE_DIR / 'lightgbm_shap_lime_top15_agreement.png'
plt.savefig(plot_path, dpi=200)
plt.show()

print('Agreement graph saved to:', plot_path)
